In [1]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "pyarrow", "pandas", "pytz", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', 'pytz', '-q'], returncode=0)

In [2]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ACCOUNT_NAME = os.environ["AZURE_STORAGE_ACCOUNT_NAME"]
ACCOUNT_KEY  = os.environ["AZURE_STORAGE_ACCOUNT_KEY"]
CONTAINER    = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")

spark = (SparkSession.builder
    .appName("Train_Model")
    .master("spark://spark-master:7077")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.pyspark.python", "/usr/bin/python3")
    .config("spark.executorEnv.PYSPARK_PYTHON", "/usr/bin/python3")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-azure:3.3.4,"
            "io.delta:delta-spark_2.13:4.0.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config(f"spark.hadoop.fs.azure.account.key.{ACCOUNT_NAME}.blob.core.windows.net", ACCOUNT_KEY)
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
BASE_PATH = f"wasbs://{CONTAINER}@{ACCOUNT_NAME}.blob.core.windows.net"
GOLD_PATH = f"{BASE_PATH}/gold/training_features"
print(f"Spark version: {spark.version}")

Spark version: 4.0.0


# Gold, tách Train/Val/Test

In [3]:
df_gold = spark.read.format("delta").load(GOLD_PATH)

# Train: 01/2024 - 09/2025
df_train = df_gold.filter(
    (F.col("year") == 2024) |
    ((F.col("year") == 2025) & (F.col("month") <= 9))
)

# Validation: 10-12/2025
df_val = df_gold.filter(
    (F.col("year") == 2025) & (F.col("month") >= 10) & (F.col("month") <= 12)
)

# Test: 01/2026 - nay
df_test = df_gold.filter(F.col("year") >= 2026)

n_train, n_val, n_test = df_train.count(), df_val.count(), df_test.count()
print(f"Train: {n_train:,} rows")
print(f"Val:   {n_val:,} rows")
print(f"Test:  {n_test:,} rows")
print(f"Tổng:  {n_train + n_val + n_test:,} (so với Gold: {df_gold.count():,})")

Train: 15,791,119 rows
Val:   2,271,083 rows
Test:  3,820,505 rows
Tổng:  21,882,707 (so với Gold: 21,887,219)


# Class distribution trên train

In [4]:
class_counts_df = (df_train.groupBy("future_congestion_15min")
    .count()
    .orderBy("future_congestion_15min"))

class_counts_df.show()

class_counts = {row["future_congestion_15min"]: row["count"]
                for row in class_counts_df.collect()}
total = sum(class_counts.values())
num_classes = len(class_counts)

print(f"Tổng Train: {total:,}")
print(f"Số lớp: {num_classes}")  # kỳ vọng 3 (0/1/2)
for label, cnt in sorted(class_counts.items()):
    print(f"  Class {label}: {cnt:,} rows ({cnt/total*100:.2f}%)")

+-----------------------+--------+
|future_congestion_15min|   count|
+-----------------------+--------+
|                      0|10920118|
|                      1| 2317579|
|                      2| 2553422|
+-----------------------+--------+

Tổng Train: 15,791,119
Số lớp: 3
  Class 0: 10,920,118 rows (69.15%)
  Class 1: 2,317,579 rows (14.68%)
  Class 2: 2,553,422 rows (16.17%)


# Tính class_weight = total / (3 × count_per_class)

In [5]:
class_weight_map = {
    label: total / (3 * cnt)
    for label, cnt in class_counts.items()
}
print("Class weight map:", class_weight_map)

mapping_expr = F.create_map(
    *[F.lit(x) for pair in class_weight_map.items() for x in pair]
)

df_train = df_train.withColumn(
    "class_weight",
    mapping_expr[F.col("future_congestion_15min")].cast("double")
)

df_train.select("future_congestion_15min", "class_weight").distinct() \
    .orderBy("future_congestion_15min").show()

Class weight map: {0: 0.4820191808672153, 1: 2.2712090217133194, 2: 2.0614322009183494}
+-----------------------+------------------+
|future_congestion_15min|      class_weight|
+-----------------------+------------------+
|                      0|0.4820191808672153|
|                      1|2.2712090217133194|
|                      2|2.0614322009183494|
+-----------------------+------------------+



# Bắt đầu chuẩn bị train
# Thư viện ML

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Định nghĩa feature columns

In [7]:
FEATURE_COLS = [
    # Spatial / categorical
    "link_id_idx", "borough_idx",
    # Thời gian
    "hour", "day_of_week", "month", "is_weekend", "is_holiday",
    # Speed hiện tại
    "speed_ratio", "current_congestion",
    # Lag congestion
    "past_congestion_15min", "past_congestion_30min",
    "past_congestion_45min", "past_congestion_60min",
    # Lag speed_ratio
    "past_speed_ratio_15min", "past_speed_ratio_30min",
    "past_speed_ratio_45min", "past_speed_ratio_60min",
    # Trend
    "congestion_trend", "speed_ratio_trend",
]
# Tổng: 19 features (link_id_idx + borough_idx thay thế cho raw string)

LABEL_COL   = "future_congestion_15min"
WEIGHT_COL  = "class_weight"

# Định nghĩa và chạy grid search

In [ ]:
import time

# 6 combo: numTrees=200, maxDepth ∈ {15,20,25}, minInstancesPerNode ∈ {1,2}
COMBOS = [
    {"numTrees": 200, "maxDepth": 15, "minInstancesPerNode": 1},
    {"numTrees": 200, "maxDepth": 15, "minInstancesPerNode": 2},
    {"numTrees": 200, "maxDepth": 20, "minInstancesPerNode": 1},
    {"numTrees": 200, "maxDepth": 20, "minInstancesPerNode": 2},
    {"numTrees": 200, "maxDepth": 25, "minInstancesPerNode": 1},
    {"numTrees": 200, "maxDepth": 25, "minInstancesPerNode": 2},
]

results = []

for i, combo in enumerate(COMBOS):
    print(f"\n{'='*60}")
    print(f"[{i+1}/6] numTrees={combo['numTrees']}, maxDepth={combo['maxDepth']}, "
          f"minInstancesPerNode={combo['minInstancesPerNode']}")
    t0 = time.time()

    # --- Build Pipeline ---
    indexer_link    = StringIndexer(inputCol="link_id",  outputCol="link_id_idx",  handleInvalid="keep")
    indexer_borough = StringIndexer(inputCol="borough",  outputCol="borough_idx",  handleInvalid="keep")

    assembler = VectorAssembler(
        inputCols=FEATURE_COLS,
        outputCol="features",
        handleInvalid="skip"   # skip row nếu vẫn còn null feature (safety net)
    )

    rf = RandomForestClassifier(
        labelCol=LABEL_COL,
        featuresCol="features",
        weightCol=WEIGHT_COL,
        numTrees=combo["numTrees"],
        maxDepth=combo["maxDepth"],
        minInstancesPerNode=combo["minInstancesPerNode"],
        maxBins=132,                        # cố định
        impurity="gini",                    # cố định
        featureSubsetStrategy="sqrt",       # cố định
        seed=42                             # cố định
    )

    pipeline = Pipeline(stages=[indexer_link, indexer_borough, assembler, rf])

    # --- Fit trên Train ---
    model = pipeline.fit(df_train)
    print(f"  Fit xong sau {time.time()-t0:.1f}s")

    # --- Transform Val ---
    df_val_pred = model.transform(df_val)

    # --- Tính Macro F1 thủ công ---
    # MulticlassClassificationEvaluator không có metricName="fMeasureByLabel" trực tiếp,
    # ta dùng metricLabel để lấy F1 từng class rồi tự trung bình
    f1_per_class = []
    for label_val in [0, 1, 2]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol=LABEL_COL,
            predictionCol="prediction",
            metricName="fMeasureByLabel",
            metricLabel=float(label_val),
            beta=1.0
        )
        f1 = evaluator.evaluate(df_val_pred)
        f1_per_class.append(f1)
        print(f"    Class {label_val} F1: {f1:.4f}")

    macro_f1 = sum(f1_per_class) / len(f1_per_class)
    elapsed  = time.time() - t0

    print(f"  → Macro F1 = {macro_f1:.4f}  (total {elapsed:.1f}s)")

    results.append({
        **combo,
        "macro_f1": macro_f1,
        "f1_class0": f1_per_class[0],
        "f1_class1": f1_per_class[1],
        "f1_class2": f1_per_class[2],
        "elapsed_s": round(elapsed, 1),
        "model": model   # giữ lại để dùng nếu combo này là best
    })


[1/6] numTrees=200, maxDepth=15, minInstancesPerNode=1


# In bảng kết quả, chọn best combo

In [ ]:
import pandas as pd

# In bảng kết quả
df_results = pd.DataFrame([
    {k: v for k, v in r.items() if k != "model"}
    for r in results
])
df_results = df_results.sort_values("macro_f1", ascending=False).reset_index(drop=True)
print(df_results.to_string(index=False))

# Chọn combo tốt nhất
best_result = max(results, key=lambda x: x["macro_f1"])
best_params = {k: v for k, v in best_result.items()
               if k in ("numTrees", "maxDepth", "minInstancesPerNode")}
best_macro_f1 = best_result["macro_f1"]
best_model_from_grid = best_result["model"]   # model đã fit trên Train

print(f"\n Best combo: {best_params}")
print(f"   Macro F1 trên Val: {best_macro_f1:.4f}")

# Gộp Train+Val

In [ ]:
# Gộp Train (01/2024–09/2025) + Val (10–12/2025)
df_train_val = df_train.unionByName(df_val)

# Val chưa có cột class_weight → cần thêm trước khi union
# (df_train đã có class_weight từ bước trên, df_val chưa có)
df_val_weighted = df_val.withColumn(
    "class_weight",
    mapping_expr[F.col(LABEL_COL)].cast("double")
)

df_train_val = df_train.unionByName(df_val_weighted)

n_tv = df_train_val.count()
print(f"Train+Val: {n_tv:,} rows")
print(f"  (Train: {n_train:,} + Val: {n_val:,} = {n_train + n_val:,}, check: {n_tv == n_train + n_val})")

# Refit Pipeline từ đầu trên Train+Val

In [ ]:
t0 = time.time()

# Build lại Pipeline với best params
indexer_link_tv    = StringIndexer(inputCol="link_id",  outputCol="link_id_idx",  handleInvalid="keep")
indexer_borough_tv = StringIndexer(inputCol="borough",  outputCol="borough_idx",  handleInvalid="keep")

assembler_tv = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol="features",
    handleInvalid="skip"
)

rf_best = RandomForestClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    weightCol=WEIGHT_COL,
    numTrees=best_params["numTrees"],
    maxDepth=best_params["maxDepth"],
    minInstancesPerNode=best_params["minInstancesPerNode"],
    maxBins=132,
    impurity="gini",
    featureSubsetStrategy="sqrt",
    seed=42
)

pipeline_final = Pipeline(stages=[indexer_link_tv, indexer_borough_tv, assembler_tv, rf_best])

# Fit trên Train+Val
final_model = pipeline_final.fit(df_train_val)

print(f" Refit xong sau {time.time()-t0:.1f}s")
print(f"   Best params: {best_params}")
print(f"   Trained on: {n_tv:,} rows (Train+Val)")

In [ ]:
# Chỉ để verify model không bị degenerate — eval chính thức ở bước 19 trên Test
df_val_sanity = final_model.transform(df_val)

for label_val in [0, 1, 2]:
    ev = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL,
        predictionCol="prediction",
        metricName="fMeasureByLabel",
        metricLabel=float(label_val),
        beta=1.0
    )
    print(f"  Val sanity — Class {label_val} F1: {ev.evaluate(df_val_sanity):.4f}")

# Kỳ vọng Val F1 cao hơn kết quả grid search (vì model vừa train trên Val luôn)
# → đây là overfit nhẹ bình thường, KHÔNG dùng số này báo cáo
print("\n  Số Val ở đây chỉ là sanity check, KHÔNG phải final eval — xem bước 19 (Test set)")